In [1]:
# Load packages
%load_ext autoreload
%autoreload 2
from sklearn.cluster import SpectralClustering as sc
import numpy.random as rand
from numpy.linalg import norm as dist
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from functools import partial
from scipy.special import expit
from sklearn.linear_model import Ridge as ridge
from sklearn.linear_model import Lasso as lasso
from sklearn.linear_model import LinearRegression as ols
from sklearn.svm import SVR as kernel_svr
from sklearn.svm import LinearSVR as lin_svr
from sklearn.kernel_ridge import KernelRidge as kernel_ridge

from sklearn.tree import DecisionTreeRegressor as reg_tree
from sklearn.ensemble import RandomForestRegressor as rfr
from sklearn.ensemble import AdaBoostRegressor as ada_reg
from sklearn.ensemble import GradientBoostingRegressor as gbr
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

from sklearn.model_selection import GridSearchCV

import copy

import statsmodels.api as sm
import statsmodels.formula.api as smf

rmse = lambda x,y: np.power(mse(x,y), 0.5)

import pandas as pd

import multiprocessing
_n_cpu = multiprocessing.cpu_count()
print(_n_cpu)
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri as np2ri

robjects.numpy2ri.activate()

from rpy2.robjects.packages import importr
grf = importr('grf')
cf = grf.causal_forest

import GPy
from sklearn.metrics import mean_squared_error

# Apparently you may use different seed values at each stage
seed_value= 1234
# 1. Set the `PYTHONHASHSEED` environment variable at a fixed value
import os
os.environ['PYTHONHASHSEED']=str(seed_value)
# 2. Set the `python` built-in pseudo-random generator at a fixed value
import random
random.seed(seed_value)
# 3. Set the `numpy` pseudo-random generator at a fixed value
import numpy as np
np.random.seed(seed_value)

8


In [ ]:
# Load data

import pandas as pd
star_data = pd.read_csv('/path/to/your/CausalICM/STAR_Students_for_Uri.csv')
import numpy as np
#star_data = star_data.sample(n=200)
treatment_filter = np.isfinite(star_data.g1classtype)
outcome_filter = np.isfinite(star_data.g1tlistss + star_data.g1treadss + star_data.g1tmathss)

def _to_categorical(X):
    assert(type(X) == type(np.ones((1,))))
    return pd.get_dummies(pd.DataFrame(X.astype(str)), dummy_na=True).values.astype(np.int)

cat_covar_columns = ['gender', 'race', 'birthmonth', 'birthyear', 'g1tchid', 'g1freelunch' ] 
_confounding_covar = 'g1surban'

# 'gkclasssize' same as gk and maybe g1
num_covar_columns = [ 'gktreadss', 'gktmathss', 'gktlistss']
Y_columns = [ 'g1treadss', 'g1tmathss', 'g1tlistss']


# all is post-filtering
# CHECKED
T_all = star_data.g1classtype[np.logical_and(treatment_filter, outcome_filter)].values # check
X_all = (star_data[cat_covar_columns][np.logical_and(treatment_filter, outcome_filter)].fillna(0).values[T_all!=3]) #check
print(X_all.shape)
#X_all = (star_data[cat_covar_columns][np.logical_and(treatment_filter, outcome_filter)].fillna(0).values[T_all!=3]) #check
Y_cols = star_data[Y_columns][np.logical_and(treatment_filter, outcome_filter)].values[T_all!=3] # check 
urban = star_data.g1surban[np.logical_and(outcome_filter, treatment_filter)].values[T_all!=3] # check 

rural_filter = np.logical_or(urban==1, urban==3) # 1 = inner city, 3 = rural
urban_filter = np.logical_or(urban==2, urban==4) # 2 = suburban, 4 = urban

## FLIPPING AROUND TEMPORARILY
urban_filter = np.logical_or(urban==1, urban==3) # 1 = inner city, 3 = rural
rural_filter = np.logical_or(urban==2, urban==4) # 2 = suburban, 4 = urban
rural_teach_set = set(star_data[cat_covar_columns][np.logical_and(treatment_filter, outcome_filter)][T_all!=3].g1tchid[rural_filter])
urban_teach_set = set(star_data[cat_covar_columns][np.logical_and(treatment_filter, outcome_filter)][T_all!=3].g1tchid[~rural_filter])

T_all = star_data.g1classtype[np.logical_and(treatment_filter, outcome_filter)].values[T_all!=3] # removing 2s from T. check
T_all[T_all == 2] = 0 # check. 3 is control.
Y_all = np.sum(Y_cols, axis=1)/3


# ALL SHAPES ARE AS SHOULD BE!

# checked
X_rural = X_all[rural_filter]
X_urban = X_all[urban_filter]

# checked
T_rural = T_all[rural_filter]
T_urban = T_all[urban_filter]

#checked
Y_rural = Y_all[rural_filter]
Y_urban = Y_all[urban_filter] # + 20*(2*T_all[urban_filter]-1) # checked


#checked
X_rural_control = X_rural[T_rural==0]
X_rural_treated = X_rural[T_rural==1]

#checked
X_urban_control = X_urban[T_urban==0]
X_urban_treated = X_urban[T_urban==1]

#checked
Y_rural_control = Y_rural[T_rural==0]
Y_rural_treated = Y_rural[T_rural==1]

#checked
Y_urban_control = Y_urban[T_urban==0]
Y_urban_treated = Y_urban[T_urban==1]



(4218, 6)


In [3]:
def _generate_rct_obs(X_rural, Y_rural, T_rural, X_urban, Y_urban, T_urban, rct_fraction_of_rural = 0.5, eval_fraction_of_notrct = 0.4):
    #splitting rural into rct and not rct
    # checked
    """
    X_rural -> X_rct + X_rural_not_RCT
    
    X_rural_not_RCT + X_urban -> X_notrct
    
    X_notrct -> X_eval + X_noteval
    
    X_noteval -> X_noteval_urban + X_noteval_rural
    
    X_noteval_urban is further curropted
    
    
    """
    X_rct, X_rural_not_RCT, Y_rct, Y_rural_not_RCT, T_rct, T_rural_not_rct = train_test_split(X_rural,
                                                                                              Y_rural,
                                                                                              T_rural,
                                                                                              test_size = rct_fraction_of_rural) 
    #checked
    X_notrct = np.vstack((X_rural_not_RCT, X_urban))
    Y_notrct = np.hstack((Y_rural_not_RCT, Y_urban))
    T_notrct = np.hstack((T_rural_not_rct, T_urban))
    _local_rural_filter = np.array([True]*X_rural_not_RCT.shape[0] + [False]*X_urban.shape[0]).ravel()
    assert(_local_rural_filter.shape == Y_notrct.shape)
    
    X_eval, X_noteval, Y_eval, Y_noteval, T_eval, T_noteval, _local_rural_filter_eval, _local_rural_filter_noteval = train_test_split(X_notrct, Y_notrct, T_notrct, _local_rural_filter, test_size = 1 - eval_fraction_of_notrct) 
    
    X_noteval_rural = X_noteval[_local_rural_filter_noteval]
    Y_noteval_rural = Y_noteval[_local_rural_filter_noteval]
    T_noteval_rural = T_noteval[_local_rural_filter_noteval]
    
    # checked
    X_noteval_rural_treated = X_noteval_rural[T_noteval_rural==1]
    X_noteval_rural_control = X_noteval_rural[T_noteval_rural==0]
        
    #checked 
    Y_noteval_rural_treated = Y_noteval_rural[T_noteval_rural==1]
    Y_noteval_rural_control = Y_noteval_rural[T_noteval_rural==0]
    
    # OUTCOME BASED REMOVAL
    _treated_rural_filter = Y_noteval_rural_treated<np.median(Y_noteval_rural_treated)
    _treated_urban_filter = Y_urban_treated<np.median(Y_urban[T_urban==1])

    _treated_rural_filter = Y_noteval_rural_treated<np.percentile(Y_noteval_rural_treated, q=30)
    _treated_urban_filter = Y_urban_treated<np.percentile(Y_urban[T_urban==1], q=30)

    # remove large samples.
    X_obs_treated_rural = X_noteval_rural_treated[_treated_rural_filter,:]
    Y_obs_treated_rural = Y_noteval_rural_treated[_treated_rural_filter]
    
    X_obs_treated_urban = X_urban_treated[_treated_urban_filter,:]
    Y_obs_treated_urban = Y_urban_treated[_treated_urban_filter]
    
    
    assert(X_obs_treated_rural.shape[0] == Y_obs_treated_rural.shape[0])

    #checked
    X_obs = np.vstack((X_obs_treated_rural,
                       X_obs_treated_urban,
                       X_noteval_rural_control,
                       X_urban_control))

    #checked
    Y_obs = np.vstack((Y_obs_treated_rural.reshape(-1,1),
                       Y_obs_treated_urban.reshape(-1,1),
                       Y_noteval_rural_control.reshape(-1,1),
                       Y_urban_control.reshape(-1,1))).ravel()

    #checked
    T_obs = np.array([1]*int(Y_obs_treated_rural.shape[0] + Y_obs_treated_urban.shape[0]) + 
                     [0]*int(Y_noteval_rural_control.shape[0] + Y_urban_control.shape[0]))
    
    assert(Y_eval.shape == T_eval.shape)
    assert(Y_rct.shape == T_rct.shape)
    assert(Y_obs.shape == Y_obs.shape)
    assert(X_rct.shape[0] == T_rct.shape[0])
    assert(X_obs.shape[0] == T_obs.shape[0])
    assert(X_eval.shape[0] == T_eval.shape[0])
    
    return X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval, Y_eval, T_eval

In [4]:
# Apparently you may use different seed values at each stage
seed_value= 1234
# 1. Set the `PYTHONHASHSEED` environment variable at a fixed value
import os
os.environ['PYTHONHASHSEED']=str(seed_value)
# 2. Set the `python` built-in pseudo-random generator at a fixed value
import random
random.seed(seed_value)
# 3. Set the `numpy` pseudo-random generator at a fixed value
import numpy as np
np.random.seed(seed_value)

rct_fraction_of_rural = 0.7
eval_fraction_of_notrct = 0.1

X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval, Y_eval, T_eval = _generate_rct_obs(X_rural,
                                                                                     Y_rural,
                                                                                     T_rural,
                                                                                     X_urban,
                                                                                     Y_urban,
                                                                                     T_urban,
                                                                                     rct_fraction_of_rural = rct_fraction_of_rural,
                                                                                     eval_fraction_of_notrct = eval_fraction_of_notrct)

In [5]:
print("RCT sample size: ", X_rct.shape[0])
print("Observational sample size: ", X_obs.shape[0])
print("Evaluation set sample size: ", X_eval.shape[0])

RCT sample size:  422
Observational sample size:  2593
Evaluation set sample size:  379


In [7]:
rwd_rct_SY_df = pd.DataFrame(np.c_[X_rct, T_rct, Y_rct])
rwd_obs_SY_df = pd.DataFrame(np.c_[X_obs, T_obs, Y_obs] )
rwd_eval_SY_df = pd.DataFrame(np.c_[X_eval, T_eval, Y_eval])
rwd_rct_SY_df.to_csv('rwd_rct_SY.csv', index=False)
rwd_obs_SY_df.to_csv('rwd_obs_SY.csv', index=False)
rwd_eval_SY_df.to_csv('rwd_eval_SY.csv', index=False)